In [0]:
%run ../../configs/variables


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail;

CREATE SCHEMA IF NOT EXISTS retail.raw;
CREATE SCHEMA IF NOT EXISTS retail.bronze;
CREATE SCHEMA IF NOT EXISTS retail.silver;
CREATE SCHEMA IF NOT EXISTS retail.gold;

CREATE VOLUME IF NOT EXISTS retail.raw.landing_zone;

USE retail.raw;

In [0]:
def load_configs():
    import json
    with open('../../configs/raw_config.json', 'r') as f:
        config_dict = json.load(f)
    return config_dict

In [0]:
def load_csv(conf, file_path):
    """
    Loads a CSV or TSV file into a Spark DataFrame using the provided configuration.
    Writes to the table.
    """
    schema_str = ', '.join([f'{col} {dtype}' for col, dtype in conf['schema'].items()])
    table_nm = f"retail.raw.{conf['table_name']}"
    if  conf['file_format'] in ('csv', 'tsv'):
        df = spark.read.format('csv')\
            .option('header', conf['header']) \
            .option('sep', conf['delimiter']) \
            .schema(schema_str) \
            .load(file_path)
    else:
        df = spark.read.csv(file_path, schema = schema_str)

    print(f'Writing to table {table_nm}\n\n')

    df.write.format("delta").mode("overwrite").saveAsTable(table_nm)

In [0]:
def load_sql(conf, file_path):
    """
    Executes a SQL query provided in the configuration or from a file and writes the result to the table.
    """
    try:
        spark.sql("""CREATE TABLE IF NOT EXISTS Refunds (  
                                    refund_id INT PRIMARY KEY,  
                                    payment_id INT NOT NULL,  
                                    refund_timestamp TIMESTAMP NOT NULL,  
                                    refund_amount DECIMAL(10, 2) NOT NULL,  
                                    refund_reason STRING NOT NULL
                                )""")

        spark.sql("""INSERT OVERWRITE Refunds (refund_id, payment_id, refund_timestamp, refund_amount, refund_reason)  
                                        VALUES  
                                        (1, 66, '2025-01-10 11:30:00', 85.75, 'Payment Error:Retailer'),  
                                        (2, 69, '2025-01-03 12:40:15', 120.50, 'Order Cancelled:Customer'),  
                                        (3, 72, '2025-01-06 14:45:30', 65.00, 'Product Returned:Customer'),  
                                        (4, 73, '2025-01-07 16:10:45', 210.99, 'Order Cancelled:Customer'),  
                                        (5, 75, '2025-01-09 18:25:00', 45.20, 'Payment Error:Retailer'),  
                                        (6, 80, '2025-01-10 09:35:20', 130.15, 'Order Cancelled:Customer'),  
                                        (7, 83, '2025-01-12 11:20:40', 150.00, 'Product Returned:Customer'),  
                                        (8, 85, '2025-01-14 13:15:30', 89.99, 'Order Cancelled:Customer'),  
                                        (9, 89, '2025-01-15 15:00:00', 78.50, 'Payment Error:Retailer'),  
                                        (10, 91, '2025-01-17 16:45:15', 250.75, 'Product Returned:Customer')
                                        """)
    except Exception as e :
        print('Error occured', e)

In [0]:
def load_json(conf, file_path):
    """
    Loads a JSON file into a Spark DataFrame using the provided configuration.
    Writes to the table.
    """
    schema_str = ', '.join([f'{col} {dtype}' for col, dtype in conf['schema'].items()])
    df = spark.read.format('json') \
        .schema(schema_str) \
        .load(file_path)
    
    table_nm = f"retail.raw.{conf['table_name']}"

    print(f'Writing to table {table_nm}\n\n')
    df.write.format("delta").mode("overwrite").saveAsTable(table_nm)


In [0]:
def load_raw_data(configs):
    """
    Loads raw data files based on the provided configuration dictionary.
    Supports CSV, TSV, and JSON file formats.
    """
    for conf in configs:
        file_path = landing_volume_path + conf['file_name']
        print(f'Loading {file_path}:')
        if conf['file_format'] in ('tsv', 'csv','excel'):
            load_csv(conf, file_path)
        elif conf['file_format'] == 'json':
            load_json(conf, file_path)
        elif conf['file_format'] == 'sql':
            load_sql(conf, file_path)
        else:
            print(f'File format {conf["file_format"]} not supported')


In [0]:
if __name__ == '__main__':
  print("-"*70,"\n","Loading data from landing zone to RAW layer(retail.raw schema)","\n","-"*70)
  configs = load_configs()
  load_raw_data(configs) 

# Loaded Tables

In [0]:
%sql
Table retail.raw.refunds limit 5

In [0]:
%sql
Table retail.raw.payments limit 5

In [0]:
%sql
Table retail.raw.orders limit 5

In [0]:
%sql
Table retail.raw.customers limit 5

In [0]:
%sql
Table retail.raw.addresses limit 5